### **Premisa**: Gonzalo recibio una herencia importante y quiere invertir el dinero en algo que genere capital de manera sostenida para no tener que trabajar mas. Una de las recomendaciones que recibio, ya que vive en BsAs, es iniciar una carrera como 'empresario Airbnb'. Esto suena bien, pero la decision de si es un buen negocio y de que forma encararlo (# de propiedades para comprar, tipo, ubicacion) parece dificil. Con el objetivo de tomar una decision informada, trabajaremos con el dataset de Inside Airbnb de Buenos Aires para tratar de encontrar una formula que aumente las chances de convertirse en un empresario Airbnb **exitoso** _(aun no sabemos lo que significa en terminos del ds)_

In [ ]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st
from scipy.stats import describe
from scipy.stats import chi2_contingency, kendalltau, f_oneway, pointbiserialr
#import requests

In [ ]:
def plot_pie(data, column, ax=None, startangle=90, palette='pastel', title=None):
    counts = data[column].value_counts()
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3))
    
    ax.pie(
        counts,
        labels=counts.index,
        autopct='%1.1f%%',
        startangle=startangle,
        colors=sns.color_palette(palette)
    )
    ax.set_title(title if title else f'Distribución de "{column}"')
    ax.legend(loc="best")
    ax.axis('equal') # Asegura que el gráfico sea circular

In [ ]:
# Función para graficar
def plot_histograma(data, column, figsize=(6, 3), bins=15, kde=True, mvd=True, shade=True, snk=False):
    skewness = (data[column]).skew()
    kurtosis = (data[column]).kurt()
    media = (data[column]).mean()
    var = (data[column]).var()
    std = (data[column]).std()
    plt.figure(figsize=figsize)
    plt.grid(axis='y')
    sns.histplot(data[column], bins=bins, kde=kde)
    if snk:
        plt.figtext(0.7, 0.8, f'Asimetría: {skewness:.2f}', fontsize=10, color='blue')
        plt.figtext(0.715, 0.73, f'Curtosis: {kurtosis:.2f}', fontsize=10, color='blue')
        plt.axvline(media, color='red', linestyle='--', label='Media')

1. Cargamos el dataset de 'listings'. Hay uno resumido y otro extendido. El resumido parece mas util

In [ ]:
df = pd.read_csv("listings_ext.csv")

2. Primera idea de que hay

In [ ]:
pd. set_option('display.max_columns', None)
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include=['number']).shape[1]
str_cols = df.select_dtypes(include=['object', 'string']).shape[1]

print("Atributos numericos:", num_cols)
print("Atributos string:", str_cols)

27348 obs, 85 atributos, la mayoria numericos pero posiblemente muchos de ellos de poco interes. Por ello hay que hacer EDA

In [ ]:
df.info()

3. Contamos cantidad de nulos

In [ ]:
pd.options.display.min_rows = 115
print(df.isnull().sum(axis=0).sort_values(ascending=False))

Eliminamos los atributos vacios

In [ ]:
df3_1 = df.drop(columns=['calendar_updated', 'host_since', 'neighbourhood_group_cleansed','neighbourhood','host_verifications','host_total_listings_count','host_neighbourhood','host_thumbnail_url','host_acceptance_rate','host_response_rate','host_response_time','estimated_revenue_l365d', 'price', 'neighborhood_overview', 'instant_bookable', 'license', 'host_about'])
df3_1.head()

In [ ]:
df3_1.shape

Eliminamos otras columnas que parecen irrelevantes a nuestro problema. Se eliminan todas la variables relacionadas a el tipo de propiedad que tienen cada huesped, si tienen o no foto, la cantidad de propiedades por huesped

In [ ]:
pd.options.display.max_columns = 85
df3_2 = df3_1.drop(columns=['host_has_profile_pic', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes','calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms','reviews_per_month', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights','minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'bathrooms_text', 'minimum_minimum_nights', 'name', 'description', 'host_name', 'first_review', 'last_review', 'host_url', 'last_scraped', 'listing_url', 'scrape_id', 'source','picture_url','host_profile_id','host_profile_url','host_location','host_picture_url','calendar_last_scraped'])#,'host_thumbnail_url'])#
df3_2.head()

Vamos a verificar la correlacion entre todas las variables numericas

In [ ]:
# Seleccionar solo las columnas numéricas de la lista
# numerical_cols = df3_2.select_dtypes(include=['number']).columns

# Calcular matriz de correlación Spearman
# corr = df3_2[numerical_cols].corr(method='spearman')

# Graficar
# plt.figure(figsize=(16, 14))
# sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, 
#            annot_kws={"size": 7}, cbar_kws={"shrink": 0.8})
# plt.title("Matriz de correlación - variables numéricas", fontsize=14)
# plt.xticks(rotation=45, ha='right')
# plt.tight_layout()
# plt.show()

<!-- Reducción de dimensionalidad por correlación entre variables numéricas

A partir de la matriz de correlación de Spearman, identificamos tres grupos de 
variables altamente correlacionadas entre sí, lo que indica redundancia de información:

**Grupo 1 - Review scores**: `review_scores_rating`, `accuracy`, `cleanliness`, 
`checkin`, `communication` y `value` presentan correlaciones superiores a 0.55 entre sí. 
Conservamos `review_scores_rating` como representante del grupo (es la variable con 
correlaciones más altas y consistentes con el resto) y `review_scores_location`, 
que mantiene correlaciones bajas (~0.44) y por lo tanto aporta información distinta.

**Grupo 2 - Availability**: `availability_30`, `availability_90`, `availability_365` 
y `availability_eoy` presentan correlación moderada-alta con `availability_60` 
(mínimo 0.52). Conservamos únicamente `availability_60` como representante del grupo.

**Grupo 3 - Reviews/Occupancy**: `number_of_reviews`, `number_of_reviews_l30d`, 
`number_of_reviews_ly` y `estimated_occupancy_l365d` presentan correlación de al 
menos 0.66 con `number_of_reviews_ltm`. Conservamos `number_of_reviews_ltm` como 
representante del grupo.

Con esta reducción pasamos de 13 variables potencialmente redundantes a 4, 
disminuyendo la dimensionalidad sin perder información relevante para el análisis. -->

In [ ]:
# columnas_eliminar = [
#     # Grupo 1 - Review scores
#     'review_scores_accuracy',
#     'review_scores_cleanliness',
#     'review_scores_checkin',
#     'review_scores_communication',
#     'review_scores_value',
    
#     # Grupo 2 - Availability
#     'availability_30',
#     'availability_90',
#     'availability_365',
#     'availability_eoy',
    
#     # Grupo 3 - Reviews/Occupancy
#     'number_of_reviews',
#     'number_of_reviews_l30d',
#     'number_of_reviews_ly',
#     'estimated_occupancy_l365d',
# ]

# df3_2 = df3_2.drop(columns=columnas_eliminar).copy()
# print(df3_2.shape)
# print(df3_2.columns.tolist())

In [ ]:
# df3_2.describe()

### Variables numericas

| Variable | Descripción | Tipo | Observación |
|---|---|---|---|
| `id` | Identificador del alojamiento | Discreta | Identificador, sin sentido aritmético |
| `hosts_time_as_user_years` | Cantidad de años de host como usuario | Discreta | Toma valores enteros positivos |
| `hosts_time_as_user_months` | Cantidad de meses de host como usuario (unificada) | Discreta | Toma valores enteros positivos |
| `hosts_time_as_host_years` | Cantidad de años como host | Discreta | Toma valores enteros positivos |
| `hosts_time_as_host_months` | Cantidad de meses como host (unificada) | Discreta | Toma valores enteros positivos |
| `host_listings_count` | Cantidad de propiedades que tiene publicadas el usuario | Discreta | Toma valores enteros positivos, posible outlier (max 945) |
| `latitude` | Coordenada geográfica norte-sur del alojamiento | Continua | Coordenada geoespacial |
| `longitude` | Coordenada geográfica este-oeste del alojamiento | Continua | Coordenada geoespacial |
| `accommodates` | Número máximo de huéspedes en una estadía | Discreta | Toma valores enteros positivos |
| `bathrooms` | Cantidad de baños en la propiedad | Discreta | Toma valores enteros positivos, nulos (0.03%) |
| `bedrooms` | Cantidad de dormitorios en la propiedad | Discreta | Toma valores enteros positivos, nulos (0.22%) |
| `beds` | Cantidad de camas en la propiedad | Discreta | Toma valores enteros positivos, nulos (0.03%) |
| `minimum_nights` | Cantidad mínima de noches | Discreta | Toma valores enteros positivos, nulos (0.39%) |
| `maximum_nights` | Cantidad máxima de noches | Discreta | Toma valores enteros positivos, nulos (0.39%) |
| `availability_30` | Disponibilidad en 30 dias | Discreta | Toma valores enteros positivos |
| `availability_60` | Disponibilidad en 60 días | Discreta | Representante del grupo de availability (alta correlación con 30/90/365/eoy) |
| `availability_90` | Disponibilidad en 90 dias | Discreta | Toma valores enteros positivos |
| `availability_365` | Disponibilidad en 365 dias | Discreta | Toma valores enteros positivos |
| `availability_eoy` | Disponibilidad al fin del anio | Discreta | Toma valores enteros positivos |
| `number_of_reviews` | Cantidad de reviews del listing | Discreta | Toma valores enteros positivos |
| `number_of_reviews_l30d` | Cantidad de reviews ultimos 30 dias | Discreta | Toma valores enteros positivos |
| `number_of_reviews_ly` | Cantidad de reviews ultimo anio | Discreta | Toma valores enteros positivos |
| `number_of_reviews_l365d` | Cantidad de reviews ultimos 365 dias | Discreta | Toma valores enteros positivos |
| `number_of_reviews_ltm` | Cantidad de reseñas en el último año | Discreta | Representante del grupo de reviews (alta correlación con number_of_reviews, l30d, ly, occupancy) |
| `review_scores_rating` | Puntaje general del listing | Continua | Valores entre 1 y 5, nulos (12.08%) |
| `review_scores_location` | Puntaje de ubicación del listing | Continua | Valores entre 1 y 5, nulos (12.08%), baja correlación con el resto |
| `review_scores_accuracy` | Precio por noche | Numerica continua| Toma valores enteros entre 1 y 5 |
| `review_scores_cleanliness` | Precio por noche | Numerica continua| Toma valores enteros entre 1 y 5 |
| `review_scores_checkin` | Precio por noche | Numerica continua| Toma valores enteros entre 1 y 5 |
| `review_scores_communication` | Precio por noche | Numerica continua| Toma valores enteros entre 1 y 5 |
| `review_scores_value` | Precio por noche | Numerica continua| Toma valores enteros entre 1 y 5 |

### Variables categoricas

| Variable | Descripción | Tipo | Observación |
|---|---|---|---|
| `id` | Identificador del alojamiento | Discreta | Identificador, sin sentido aritmético |
| `hosts_time_as_user_years` | Cantidad de años de host como usuario | Discreta | Toma valores enteros positivos |
| `hosts_time_as_user_months` | Cantidad de meses de host como usuario (unificada) | Discreta | Toma valores enteros positivos |
| `hosts_time_as_host_years` | Cantidad de años como host | Discreta | Toma valores enteros positivos |
| `hosts_time_as_host_months` | Cantidad de meses como host (unificada) | Discreta | Toma valores enteros positivos |
| `host_listings_count` | Cantidad de propiedades que tiene publicadas el usuario | Discreta | Toma valores enteros positivos, posible outlier (max 945) |
| `latitude` | Coordenada geográfica norte-sur del alojamiento | Continua | Coordenada geoespacial |
| `longitude` | Coordenada geográfica este-oeste del alojamiento | Continua | Coordenada geoespacial |
| `accommodates` | Número máximo de huéspedes en una estadía | Discreta | Toma valores enteros positivos |
| `bathrooms` | Cantidad de baños en la propiedad | Discreta | Toma valores enteros positivos, nulos (0.03%) |
| `bedrooms` | Cantidad de dormitorios en la propiedad | Discreta | Toma valores enteros positivos, nulos (0.22%) |
| `beds` | Cantidad de camas en la propiedad | Discreta | Toma valores enteros positivos, nulos (0.03%) |
| `minimum_nights` | Cantidad mínima de noches | Discreta | Toma valores enteros positivos, nulos (0.39%) |
| `maximum_nights` | Cantidad máxima de noches | Discreta | Toma valores enteros positivos, nulos (0.39%) |
| `availability_60` | Disponibilidad en 60 días | Discreta | Representante del grupo de availability (alta correlación con 30/90/365/eoy) |
| `number_of_reviews_ltm` | Cantidad de reseñas en el último año | Discreta | Representante del grupo de reviews (alta correlación con number_of_reviews, l30d, ly, occupancy) |
| `review_scores_rating` | Puntaje general del listing | Continua | Valores entre 1 y 5, nulos (12.08%) |
| `review_scores_location` | Puntaje de ubicación del listing | Continua | Valores entre 1 y 5, nulos (12.08%), baja correlación con el resto |

Creamos nuevos features con el tiempo del host como usuario y como host en meses para poder usarlos en analisis, y ademas tomamos la variable target host_is_superhost, para cambiaar los valors 't'= 1 y 'f'=0

In [ ]:
df3_3 = df3_2
df3_3['hosts_time_as_user_months'] = df3_3['hosts_time_as_user_years'] * 12 + df3_3['hosts_time_as_user_months']
df3_3['hosts_time_as_host_months'] = df3_3['hosts_time_as_host_years'] * 12 + df3_3['hosts_time_as_host_months']
df3_3 = df3_3.drop(columns=['hosts_time_as_user_years', 'hosts_time_as_host_years']).copy()
df3_3["host_is_superhost"] = df["host_is_superhost"].map({'t': 1, 'f': 0})
df3_3.head()

<!-- Vamos a analizar la relacon entre las varibles 'room_type' y 'property_type'.  Como vemos room_type es una agrupamiento de las caracteristicas de property_type. Por lo tanto vamos a eliminar la columna property_type -->

In [ ]:
# df3_3[['room_type', 'property_type']].value_counts()

<!-- Hacemos una verifiacion por CremmerV  para ver si mantiene realcion con el target -->

In [ ]:
# def cramers_v(x, y):
#     ct = pd.crosstab(x, y)
#     chi2 = chi2_contingency(ct)[0]
#     n = ct.sum().sum()
#     min_dim = min(ct.shape) - 1
#     return np.sqrt(chi2 / (n * min_dim))

# v_room = cramers_v(df3_3['room_type'], df3_3['host_is_superhost'])
# print(f"Cramér's V room_type vs target: {v_room:.4f}")

<!-- Entonces eliminamos la variable property_type -->

In [ ]:
# df3_3 = df3_3.drop(columns=['property_type']).copy()


Los amenities son un atributo muy interesante porque nos puede guiar con respecto a que propiedades comprar o como equiparlas. Sin embargo, ese atributo tiene demasiadas palabras. Vamos a encontrar el top 10 de palabras mas frecuentes y considerar si vale la pena crear nuevos atributos con ellas
Concatenamos todo el texto en una sola linea, lo pasamos a minuscula, contamos la frecuencia de palabras, imprimimos, filtramos palabras que no son de utilidad y repetimos el proceso sin las palabras filtradas hasta que el top 10 son palabras utiles

In [ ]:
from collections import Counter
import re

# Combine all text in the column into one big string
texto = " ".join(df3_3["amenities"].astype(str))

# Convert to lowercase and extract words
palabras = re.findall(r"\b\w+\b", texto.lower())

borrar = ['and', 'allowed', 'water', 'hot', 'clothing', 'in', 'dishes', 'silverware', 'heating', 'basics', 'refrigerator', 'cooking', 'dining', 'hangers', 'maker', 'bed']

palabras = [w for w in palabras if w not in borrar]

# Count word frequencies
frecuancias = Counter(palabras)

# Get the top 10
top_10 = frecuancias.most_common(10)

print(top_10)

In [ ]:
top_10_df = pd.DataFrame(top_10, columns=['palabra', 'frecuencia'])

plt.figure(figsize=(10, 6))
sns.barplot(data=top_10_df, x='frecuencia', y='palabra', 
            hue='palabra', legend=False, palette='pastel')
plt.title('Top 10 palabras más frecuentes en amenities')
plt.xlabel('Frecuencia')
plt.ylabel('Palabra')
plt.tight_layout()
plt.show()

Ahora creamos features binarios con las mejores palabras

In [ ]:
df3_3["coffee"] = df3_3["amenities"].str.contains("coffee", case=False, na=False).astype(int)
df3_3["wifi"] = df3_3["amenities"].str.contains("wifi", case=False, na=False).astype(int)
df3_3["parking"] = df3_3["amenities"].str.contains("parking", case=False, na=False).astype(int)
df3_3["air_conditioning"] = df3_3["amenities"].str.contains("conditioning", case=False, na=False).astype(int)
df3_3["pool"] = df3_3["amenities"].str.contains("pool", case=False, na=False).astype(int)
df3_3["gym"] = df3_3["amenities"].str.contains("gym", case=False, na=False).astype(int)
df3_3 = df3_3.drop(columns=['amenities']).copy()
df3_3.head()

In [ ]:
df3_3.describe()

Si observamos los datos para valores minimos, se ve que hay listings sin banios, habitaciones o camas. Esto puede ser tanto errores MAR como listings de arreglos poco ortodoxos. 
Vamos a anlizar la catidad de cada uno de ellos, por lo tanto habitacion = 0  puede ser un monoambiente y son el 12,79% de las variables. En el caso de bed y bathroom representan el 2,63% y 0,43%

In [ ]:
variables = ['bathrooms', 'beds']

for var in variables:
    ceros = (df3_3[var] == 0).sum()
    nulos = df3_3[var].isnull().sum()
    total = len(df3_3)
    print(f"{var}:")
    print(f"  == 0:   {ceros} registros ({ceros/total*100:.2f}%)")
    print(f"  NaN:    {nulos} registros ({nulos/total*100:.2f}%)")
    print()

In [ ]:
print(df3_3[df3_3['beds'] == 0]['room_type'].value_counts())

In [ ]:
df3_4 = df3_3[df3_3["bathrooms"].notna() & (df3_3["bathrooms"] != 0)]
df3_4 = df3_4[df3_4["beds"].notna() & (df3_4["beds"] != 0)]

df3_4.describe()

In [ ]:
df3_4.shape

4. **Analisis grafico de atributos**
Slice del dataframe para poder hacer un grafico de cantidad de Airbnb por barrio (mostrando solo los mas populares)

In [ ]:
df4_1 = df3_4
barrios_id_df = df4_1[["neighbourhood_cleansed", "id"]]
pd.options.display.min_rows = 20

Agrupamos por barrio y agregar la variable de cantidad de veces que el barrio es observado

In [ ]:
barrios_id_df =  df4_1.groupby(['neighbourhood_cleansed'])["id"].count().reset_index(name='Count').sort_values(['Count'], ascending=False)
barrios_id_df =  barrios_id_df[barrios_id_df["Count"] > 1000]
barrios_id_df

Otro slice para quedarnos con los barrios con mas de 1000 Airbnb

In [ ]:
top_barrios_df = df4_1[df4_1["neighbourhood_cleansed"].isin(["Palermo", "Recoleta", "San Nicolas", "Belgrano", "Retiro", "Monserrat", "Balvanera", "Almagro"])]

Multiples graficos de frecuencia y derivados

In [ ]:
sns.countplot(x="neighbourhood_cleansed", data=top_barrios_df, hue="neighbourhood_cleansed", palette='pastel', legend='brief')

plt.suptitle('Barrios con > 1000 Arbnb - cantidad por barrio', fontsize=14)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 8))
sns.scatterplot(data=top_barrios_df, x='longitude', y='latitude', hue='neighbourhood_cleansed', palette='pastel', alpha=0.7, s=35, edgecolor='none')
plt.title('Ubicación de barrios con > 1000 Arbnb')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Barrio')
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
df4_1.describe()

Vamos a analizar la variable 'host_listings_count' que tiene un valor maximo de 562.

In [ ]:
p99 = df4_1['host_listings_count'].quantile(0.99)
datos = df4_1[df4_1['host_listings_count'] <= p99]['host_listings_count']

plt.figure(figsize=(10, 6))
sns.histplot(datos, bins=30, kde=True, color='skyblue')
plt.title('Distribución de host_listings_count (hasta p99)')
plt.xlabel('Cantidad de propiedades del anfitrión')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

print(df4_1['host_listings_count'].describe(percentiles=[.5, .75, .90, .95, .99]))

por lo que se observa hay un 1% de valores entre 183 y 562, son valores grandes pero no atipicos. Posiblemente sean empresas que manejan alojamientos y posiblemente tengan varios alojamientos superhost

In [ ]:
df4_1.describe()

Analizamos los barrios con mayor cantidad de reviews

In [ ]:
barrios_max = df4_1.groupby('neighbourhood_cleansed')["number_of_reviews_ltm"].count().sort_values(ascending=False).head(10)
top = barrios_max.reset_index(name='count')

plt.figure(figsize=(10,6))
sns.barplot(data=top, x='neighbourhood_cleansed', y='count', palette='pastel', 
            hue='neighbourhood_cleansed', legend=False, order=top['neighbourhood_cleansed'])
plt.title('Top 10 barrios por cantidad de reviews')
plt.xlabel('Barrio')
plt.ylabel('Cantidad de reviews')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Verificamos que barrios tienen mas disponibilidad

In [ ]:
orden = df4_1.groupby('neighbourhood_cleansed')['availability_60'].median().sort_values(ascending=False).head(10).index
sns.boxplot(data=df4_1[df4_1['neighbourhood_cleansed'].isin(orden)], x='availability_60', y='neighbourhood_cleansed', order=orden)

## esto vamoa hacerlo por OHE despues del split yo tengo un csv con las comunas y ahi lo reducimos a 14 variable y lee podemos hacer un OHE

<!-- Nos quedamos con los top 5 barrios y creamos atributos categoricos binarios para cada uno -->

In [ ]:
#df4_1["Palermo"] = df4_1["neighbourhood_cleansed"].str.contains("Palermo", case=False, na=False).astype(int)
#df4_1["Recoleta"] = df4_1["neighbourhood_cleansed"].str.contains("Recoleta", case=False, na=False).astype(int)
#df4_1["San Nicolas"] = df4_1["neighbourhood_cleansed"].str.contains("San Nicolas", case=False, na=False).astype(int)
#df4_1["Belgrano"] = df4_1["neighbourhood_cleansed"].str.contains("Belgrano", case=False, na=False).astype(int)
#df4_1["Retiro"] = df4_1["neighbourhood_cleansed"].str.contains("Retiro", case=False, na=False).astype(int)
#df4_2 = df4_1.drop(columns=['neighbourhood_cleansed']).copy()
#df4_2.head()


In [ ]:
df4_2 = df4_1

Grafico de disponibilidad

In [ ]:
sns.histplot(df4_2["availability_60"].dropna(), bins=40)
plt.title("Availability 60")
plt.xlabel("availability_60")
plt.ylabel("Frecuencia")
plt.show()

Cantidad de reviews vs disponibilidad

In [ ]:
sns.scatterplot(data=df4_2, x='minimum_nights', y='number_of_reviews_ltm', color='b',alpha=0.6)
	
plt.grid(ls='--')
plt.title('')
plt.tight_layout()
plt.show()

In [ ]:
#plot_pie(Airbnb_df, "property_type")
top10 = df4_2["room_type"].value_counts().head(4).index
plot_prop_type = df4_2[df4_2["room_type"].isin(top10)]
colores = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    
]

labels = plot_prop_type["room_type"].value_counts().index
patches, texts = plt.pie(plot_prop_type["room_type"].value_counts(), colors=colores, startangle=90)
plt.title(f'Distribución de room_type')
plt.legend(patches, labels, loc="best")
# Set aspect ratio to be equal so that pie is drawn as a circle.
plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
df4_2.describe()

Analizamos la distribucion de las variables

In [ ]:
variables = ['hosts_time_as_user_months', 'hosts_time_as_host_months', 
              'host_listings_count', 'accommodates', 'bathrooms', 'bedrooms', 
              'beds', 'minimum_nights', 'maximum_nights', 'availability_60', 
              'number_of_reviews_ltm', 'review_scores_rating', 'review_scores_location']

print("=== TABLA DE PERCENTILES ===\n")
for var in variables:
    print(f"--- {var} ---")
    print(df4_2[var].describe(percentiles=[.25, .50, .75, .90, .95, .99]))
    print()

# Histogramas recortados por p99
fig, axes = plt.subplots(7, 2, figsize=(14, 22))
axes = axes.flatten()

for i, var in enumerate(variables):
    p99 = df4_2[var].quantile(0.99)
    datos = df4_2[df4_2[var] <= p99][var].dropna()
    
    sns.histplot(datos, bins=30, kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Distribución de {var} (hasta p99)')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Frecuencia')

# Ocultar el último subplot vacío (13 variables en grilla de 14)
axes[-1].set_visible(False)

plt.suptitle('Distribución de variables numéricas (sin outliers superiores)', fontsize=14)
plt.tight_layout()
plt.show()

* Se analizan todas las variables y no hay nada atipico que reportar


Disponibilidad dependiendo del tipo de alojamiento

In [ ]:
plt.figure(figsize=(20, 4))
sns.boxplot(data=plot_prop_type, x='room_type', y='availability_60', 
            hue='room_type', legend=False, palette='pastel')
plt.tight_layout()
plt.show()

Vemos l;a distribucion de otras 3 variables

In [ ]:
sns.boxplot(x=df4_2["number_of_reviews"])
plt.title("Distribuicon de number_of_reviews")
plt.show()

In [ ]:
sns.boxplot(x=df4_2["host_listings_count"])
plt.title("Distribuicon de host_listings_count")
plt.show()

In [ ]:
sns.boxplot(x=df4_2["review_scores_rating"])
plt.title("Distribuicon de review_scores_rating")
plt.show()

Observamos una gran cantidad de outliers, pero esto se debe a la naturaleza del dataset y no un error en el dataset

Vemos si los hosts con mas propiedades tienen a tener mejor puntaje de reviews o si la experiencia les juega en contra

In [ ]:
variables = ['review_scores_rating', 'host_listings_count']
corr = df4_2[variables].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlación - variables numéricas")
plt.show()

* Vemos que la correlacion es debil y negativa lo cual indicaria que cuanto mas propiedades tiene un host, este tiend a dedicar menos tiempo a cada propiedad y eso lleva a obtener peores reviews
* Y que ocurre con la relacion entre los amenities y el puntaje? Estamos buscando correlacion entre un feature categorico binario y uno numerico -> punto biserial

In [ ]:
num_cols    = ['review_scores_rating', 'number_of_reviews_ltm']
binarias = ['coffee',	'wifi',	'parking',	'air_conditioning', 'pool', 'gym']

pb_mat = pd.DataFrame(index=num_cols, columns=binarias, dtype=float)

for num_col in num_cols:
    for bin_col in binarias:
        mask = df4_2[num_col].notna() & df4_2[bin_col].notna()
        num  = df4_2.loc[mask, num_col]
        bn   = df4_2.loc[mask, bin_col]
        r, p = pointbiserialr(bn, num) # pointbiserialr Devuelve r y p-value, pero solo nos interesa r
        pb_mat.loc[num_col, bin_col] = abs(r)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(pb_mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
            vmin=0, vmax=1, linewidths=0.5, ax=ax)

ax.set_title("Punto Biserial — Asociación numérica-binaria", fontsize=11)
ax.set_xlabel("Var. binarias")
ax.set_ylabel("Var. numéricas")
ax.set_xticklabels(ax.get_xticklabels(), fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)
plt.tight_layout()
plt.show()

* Solo hay una correlacion debil entre los puntajes y el atributo cafe, con lo cual seria bueno equipar los AirBnB con caferetas

## Pueden poner un pequenio resumen de EDA aca?

**Ya tenemos el dataset con los atributos relevantes y el target listo para su procesamiento y limpieza**

Primero estudiamos la distribucion del target

In [ ]:
counts = df4_2["host_is_superhost"].value_counts()

color_map = {
    1: "green",
    0: "red"
}

colors = [color_map[v] for v in counts.index]

counts.plot(
    kind="pie",
    autopct="%1.1f%%",
    startangle=90,
    colors=colors
)

plt.title("Superhosts")
plt.ylabel("")
plt.show()

Venos que es un problema de clasificacion balanceado

In [ ]:
df4_2.head()

In [ ]:
df4_2.shape

#### Vamos a realizar el train_test_split

In [ ]:
y = df4_2['host_is_superhost']
X = df4_2.drop(columns=['host_is_superhost']).copy()

Verificamos filas y columnas de X e y

In [ ]:
X.shape, y.shape

ahora vamos a realizar el train test split asi podemos realizar todas las traformaciones en el train y luego la aplicamos en el test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y )

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

## Reduccion de dimensionalidad

Ya quitamos algunas variables redundantes y sin datos y creamos otras que son utiles para el problema de ML (cambiar el atributo origical de bariro como string a variables binarias)

## Seleccion de atributos

Primero buscamos filtrar atributos redundantes (muy correlcionados entre si) o que puedan causar problemas de multicolinealidad

Ralizamos una matriz de correlacion entre las variables numericas para capturar posibles problemas de multicolinealidad en atributos. Suponemos que correlacion de Spearman (monotonica) mayor a 0.8 es indicativo de atributos redundantes

* Por otro lado prestamos atencion a la relacion entre los distintos puntajes de reviews entre ellos y la experiencia del host para darnos una idea de atributos de un host exitoso

In [ ]:
# Seleccionar solo las columnas numéricas de la lista
numerical_cols = X_train.select_dtypes(include=['number']).columns

# Calcular matriz de correlación Spearman
corr = X_train[numerical_cols].corr(method='spearman')

# Graficar
plt.figure(figsize=(16, 14))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, 
            annot_kws={"size": 7}, cbar_kws={"shrink": 0.8})
plt.title("Matriz de correlación - variables numéricas", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Interesante que la experiencia del host como host y como usuario no esta para nada correlacionada con los puntajes de los reviews. Esto es una buena noticia porque como inversores solo podemos elegir en que invertir, pero no podemos comprar tiempo de experiencia
Resulta interesante tambien hecho de que la correlacion mas fuerte de ratings (el puntaje mas importante) es con 'accuracy' y 'value'. Esto se puede interpretar como que los usuarion valoran mucho la exactitud de la descripcion del listing y la relacion calidad-precio (value for money)

**Reducción de dimensionalidad por correlación entre variables numéricas**

A partir de la matriz de correlación de Spearman, identificamos tres grupos de 
variables altamente correlacionadas entre sí, lo que indica redundancia de información:

**Grupo 1 - Review scores**: `review_scores_rating`, `accuracy`, `cleanliness`, 
`checkin`, `communication` y `value` presentan correlaciones superiores a 0.55 entre sí. 
Conservamos `review_scores_rating` como representante del grupo (es la variable con 
correlaciones más altas y consistentes con el resto) y `review_scores_location`, 
que mantiene correlaciones bajas (~0.44) y por lo tanto aporta información distinta.

**Grupo 2 - Availability**: `availability_30`, `availability_90`, `availability_365` 
y `availability_eoy` presentan correlación moderada-alta con `availability_60` 
(mínimo 0.52). Conservamos únicamente `availability_60` como representante del grupo.

**Grupo 3 - Reviews/Occupancy**: `number_of_reviews`, `number_of_reviews_l30d`, 
`number_of_reviews_ly` y `estimated_occupancy_l365d` presentan correlación de al 
menos 0.66 con `number_of_reviews_ltm`. Conservamos `number_of_reviews_ltm` como 
representante del grupo.

Con esta reducción pasamos de 13 variables potencialmente redundantes a 4, 
disminuyendo la dimensionalidad sin perder información relevante para el análisis.

## Yo aca quitaria id y host_id porque solo agregan ruido

In [ ]:
columnas_eliminar = [
    # Grupo 1 - Review scores
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_value',
    
    # Grupo 2 - Availability
    'availability_30',
    'availability_90',
    'availability_365',
    'availability_eoy',
    
    # Grupo 3 - Reviews/Occupancy
    'number_of_reviews',
    'number_of_reviews_l30d',
    'number_of_reviews_ly',
    'estimated_occupancy_l365d',
]

X_train = X_train.drop(columns=columnas_eliminar).copy()
X_test = X_test.drop(columns=columnas_eliminar).copy()
print(X_train.shape)
print(X_train.columns.tolist())

In [ ]:
X_train.describe()

Vamos a analizar la relacon entre las varibles 'room_type' y 'property_type'.  Como vemos room_type es una agrupamiento de las caracteristicas de property_type. Por lo tanto vamos a eliminar la columna property_type

In [ ]:
X_train[['room_type', 'property_type']].value_counts()

Hacemos una verifiacion por CremmerV  para ver si mantiene realcion con el target

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    min_dim = min(ct.shape) - 1
    return np.sqrt(chi2 / (n * min_dim))

v_room = cramers_v(X_train['room_type'], y_train)
print(f"Cramér's V room_type vs target: {v_room:.4f}")

Entonces eliminamos la variable property_type

In [ ]:
X_train = X_train.drop(columns=['property_type']).copy()
X_test = X_test.drop(columns=['property_type']).copy()

#### Ahora vamos a trabajar con  de las variables categoricas

In [ ]:
X_train.describe(include='str')

Vamos a analizar la variable 'has_availavility' 

In [ ]:
X_train['has_availability'].value_counts()

'has_avalibility' es una variable constnate todo los valores son True por lo tanto no aporta valor y la vamos a eliminar

In [ ]:
X_train=X_train.drop(columns=['has_availability']).copy()
X_test=X_test.drop(columns=['has_availability']).copy()

Luego seguimos con host_identitiy_verified                      

In [ ]:
X_train['host_identity_verified'].value_counts()

Tiene varios valores esta variable y parece bastante util, ya que tener la identidad verificada deberia generar confianza en los huespedes.
Entonces vamos a hacer 't'=1 y 'f'= 0

In [ ]:
X_train['host_identity_verified'] = X_train['host_identity_verified'].map({'t': 1, 'f': 0})
X_test['host_identity_verified'] = X_test['host_identity_verified'].map({'t': 1, 'f': 0})

Ahora avanzamos con la variable 'neighbourhood_cleansed'. vemos que esta variable cuenta con 48 datos distintos, lo que vamos hacer es agruparlos por columnas por lo que nos deberia quedar de solo 14 variables y ahi podemos hacer un OHE

Importo un csv con las comunas

In [ ]:
comunas = pd.read_csv(r'barrios_comunas_caba.csv')

Verificamos que los barrios coincidan

In [ ]:
barrios_no_match = set(X_train['neighbourhood_cleansed'].unique()) - set(comunas['barrio'].unique())
print(f"Barrios sin match: {barrios_no_match}")

In [ ]:
X_train = X_train.merge(comunas, left_on='neighbourhood_cleansed', right_on='barrio', how='left')
X_train = X_train.drop(columns=['neighbourhood_cleansed', 'barrio']).copy()

# Verificar que no quedaron nulos en comuna
print(X_train['comuna'].isnull().sum())
print(X_train.shape)

In [ ]:
X_test = X_test.merge(comunas, left_on='neighbourhood_cleansed', right_on='barrio', how='left')
X_test = X_test.drop(columns=['neighbourhood_cleansed', 'barrio']).copy()

Verificamos que nos queda la variable comuna con 15 variables

In [ ]:
X_train['comuna'].unique()

Vamos  realizar el OHE a la variable 'comuna' y a la variable 'roon_type' ya que vemos que cuenta con solo 4 variables

In [ ]:
X_train

## JPC: Este codigo no me funciona:
 "None of [Index(['comuna', 'room_type'], dtype='str')] are in the [columns]"

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Definir columnas a codificar
columnas_ohe = ['comuna', 'room_type']

# Instanciar el encoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

# Fit y transform en X_train
ohe_train = ohe.fit_transform(X_train[columnas_ohe])

# Crear dataframe con las nuevas columnas
ohe_cols = ohe.get_feature_names_out(columnas_ohe)
ohe_train_df = pd.DataFrame(ohe_train, columns=ohe_cols, index=X_train.index)

# Eliminar columnas originales y agregar las codificadas
X_train = X_train.drop(columns=columnas_ohe)
X_train = pd.concat([X_train, ohe_train_df], axis=1)

X_test = X_test.drop(columns=columnas_ohe)
X_test = pd.concat([X_train, ohe_train_df], axis=1)


In [ ]:
X_train.head()

Varificamos con ya no quedan variables categoricas, y se agregaron al final las columnas del OHE.

## JPC: Esto deberia ser parte del EDA antes del split, no?

#### Vamos a verificar los valor nulos que existan

In [ ]:
X_train.isnull().sum().sort_values(ascending=False)

Vemos que los valores faltante de review_scores_rating y review_scores_location tienen la misma cantidad y vimos cuando hicimos los anlisis de variable es porque tienen o reviews.

In [ ]:
# Verificar que los nulos en review_scores coinciden con 0 reviews
print(X_train[X_train['review_scores_rating'].isnull()]['number_of_reviews_ltm'].describe())

# Ver cuántos tienen 0 reviews en total
print(f"\nTotal con number_of_reviews_ltm == 0: {(X_train['number_of_reviews_ltm'] == 0).sum()}")
print(f"Total con review_scores_rating nulo: {X_train['review_scores_rating'].isnull().sum()}")

Es decir: 100% de los nulos en review_scores corresponden a alojamientos sin reseñas recientes, confirma que el faltante es MAR (Missing At Random, condicionado a otra variable observada: no tener reviews)

Vamos la distribucion de las variables

In [ ]:
variables = ['review_scores_rating', 'review_scores_location']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, var in enumerate(variables):
    sns.histplot(X_train[var].dropna(), bins=20, kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Distribución de {var}')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Frecuencia')
    
    # Líneas de media y mediana
    media = X_train[var].mean()
    mediana = X_train[var].median()
    axes[i].axvline(media, color='red', linestyle='--', label=f'Media: {media:.2f}')
    axes[i].axvline(mediana, color='green', linestyle='--', label=f'Mediana: {mediana:.2f}')
    axes[i].legend()

plt.suptitle('Distribución de review scores (sin nulos)', fontsize=14)
plt.tight_layout()
plt.show()

Ambas variables presentan asimetría negativa (cola hacia la izquierda). En distribuciones asimétricas, la media se ve afectada por los valores extremos, 
desplazándose hacia ellos. La mediana, al ser el valor central, representa mejor el comportamiento típico y es más robusta ante estos outliers. Por lo tanto, optamos por imputar con la mediana.

Vamos a analizar las variables 'minimum_nights', 'maximum_nights' que tienen la misma cantida de ausentes

In [ ]:
variables = ['minimum_nights', 'maximum_nights']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, var in enumerate(variables):
    p99 = X_train[var].quantile(0.99)
    datos = X_train[X_train[var] <= p99][var].dropna()
    
    sns.histplot(datos, bins=30, kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Distribución de {var} (hasta p99)')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Frecuencia')
    
    media = X_train[var].mean()
    mediana = X_train[var].median()
    axes[i].axvline(media, color='red', linestyle='--', label=f'Media: {media:.2f}')
    axes[i].axvline(mediana, color='green', linestyle='--', label=f'Mediana: {mediana:.2f}')
    axes[i].legend()

plt.suptitle('Distribución de minimum_nights y maximum_nights', fontsize=14)
plt.tight_layout()
plt.show()

Vemos comportamientos distintos de la distribucion. En 'minimum_nights' se ve una distribucion asimetrica con cola hacia la derecha entonces para imputar con el menor impacto en la distribucion lo hacemos con la mediana que vemos cae donde esta la mayor poblacion. 
En el caso del 'maximum_nights' es una distgribucion multimodal con varios picos, asi que si imputamos la mediana vemos que cae en el pico mas pronunciado que es 365 que debe ser el valor mas estandar que se pone por defecto.
Conclusion en ambos graficos hay que imputar con la mediana.

Ahora analicemos la ultima que nos falta que seria bedrooms

In [ ]:
p99 = X_train['bedrooms'].quantile(0.99)
datos = X_train[X_train['bedrooms'] <= p99]['bedrooms'].dropna()

plt.figure(figsize=(8, 5))
sns.histplot(datos, bins=20, kde=True, color='skyblue')

media = X_train['bedrooms'].mean()
mediana = X_train['bedrooms'].median()
plt.axvline(media, color='red', linestyle='--', label=f'Media: {media:.2f}')
plt.axvline(mediana, color='green', linestyle='--', label=f'Mediana: {mediana:.2f}')
plt.legend()

plt.title('Distribución de bedrooms (hasta p99)')
plt.xlabel('bedrooms')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

Mismo patro y conclusiones tomadas para maximum_night. PAtron multimodal y vamos a imputar por la mediana que es quien menos nos altera la destribucion de la variable.

Concluimos que todas las variables se imputan por la mediana

In [ ]:
from sklearn.impute import SimpleImputer

columnas_imputar = ['review_scores_rating', 'review_scores_location', 
                     'minimum_nights', 'maximum_nights', 'bedrooms']

imputer = SimpleImputer(strategy='median')

X_train[columnas_imputar] = imputer.fit_transform(X_train[columnas_imputar])

print(X_train[columnas_imputar].isnull().sum())

Por ultimo corroboramos que no hay mas valores ausentes.

#### Analicemos si existen outliers

In [ ]:
X_train.describe()

Vamos analizar alguna variables que pueden taner valores maximas mas llamativos

In [ ]:
variables = ['host_listings_count', 'bathrooms', 'bedrooms', 'beds', 'number_of_reviews_ltm']

# Tabla de percentiles
print("=== TABLA DE PERCENTILES ===\n")
for var in variables:
    print(f"--- {var} ---")
    print(X_train[var].describe(percentiles=[.25, .50, .75, .90, .95, .99]))
    print()

# Histogramas recortados por p99
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

for i, var in enumerate(variables):
    p99 = X_train[var].quantile(0.99)
    datos = X_train[X_train[var] <= p99][var].dropna()
    
    sns.histplot(datos, bins=30, kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Distribución de {var} (hasta p99)')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Frecuencia')

plt.suptitle('Distribución de variables numéricas (sin outliers superiores)', fontsize=14)
plt.tight_layout()
plt.show()

Se observa en todas las variables que pueden tener algunos valores altos pero no son atipicos sino de huespedes especiales.
CONCLUSION NO SE ANULA NINGUN DATO POR OUTLIERS

In [ ]:
X_train.head()

Aunque Phi y point-biserial provienen de tradiciones estadísticas distintas (Chi-cuadrado para tablas de contingencia vs. correlación de Pearson), ambas 
se reducen matemáticamente a la misma fórmula cuando las dos variables son binarias (0/1) — Phi es simplemente Pearson aplicado a ese caso particular. 
Por eso usamos una sola función para calcular la asociación de todas las features (numéricas y binarias) contra el target.

In [ ]:
from scipy import stats

def asociacion_con_target(X, y, col):
    valores_unicos = X[col].nunique()
    
    if valores_unicos < 2:
        return {'feature': col, 'tipo': 'constante', 'asociacion': np.nan, 'p_value': np.nan}
    
    try:
        corr, pvalue = stats.pointbiserialr(y, X[col])
    except Exception as e:
        return {'feature': col, 'tipo': 'error', 'asociacion': np.nan, 'p_value': np.nan}
    
    tipo = "Phi (binaria)" if valores_unicos == 2 else "Point-biserial (numerica)"
    
    return {'feature': col, 'tipo': tipo, 'asociacion': corr, 'p_value': pvalue}


resultados = []
for col in X_train.columns:
    if col in ['id', 'host_id']:
        continue
    resultados.append(asociacion_con_target(X_train, y_train, col))

resultados_df = pd.DataFrame(resultados)
resultados_df['asociacion_abs'] = resultados_df['asociacion'].abs()
resultados_df = resultados_df.sort_values('asociacion_abs', ascending=False)


def interpretar_asociacion(row):
    valor = row['asociacion']
    pvalue = row['p_value']
    
    if pd.isna(valor):
        return "no calculable"
    
    if pvalue >= 0.05:
        return "no significativa"
    
    v = abs(valor)
    if v < 0.1:
        return "significativa - débil"
    elif v < 0.3:
        return "significativa - débil-moderada"
    elif v < 0.5:
        return "significativa - moderada-fuerte"
    else:
        return "significativa - fuerte"

resultados_df['interpretacion'] = resultados_df.apply(interpretar_asociacion, axis=1)
print(resultados_df.to_string(index=False))

## JPC: La significancia estadistica tiene como requisito que los datos tengan __distribucion normal y que la varianza sea homogenea__, creo que punto biserial funcionan siempre. __Mutual Information__ tambien deberia funcionar

Para la selección de features, utilizamos como criterio la **significancia estadística** (p-value < 0.05) de la asociación entre cada feature y el target 
`host_is_superhost`, calculada mediante point-biserial/Phi según corresponda.
A diferencia de filtrar por la magnitud del coeficiente (que descartaría prácticamente todas las variables al ser, en su mayoría, asociaciones débiles), 
este criterio responde a la pregunta de si la asociación observada es estadísticamente distinguible del azar, independientemente de su tamaño

In [ ]:
# Filtrar features con asociación estadísticamente significativa (p < 0.05)
seleccionadas = resultados_df[resultados_df['p_value'] < 0.05]['feature'].tolist()
descartadas = resultados_df[resultados_df['p_value'] >= 0.05]['feature'].tolist()

print(f"Variables seleccionadas: {len(seleccionadas)}")
print(seleccionadas)
print(f"\nVariables descartadas (p >= 0.05): {len(descartadas)}")
print(descartadas)

## Hay que aplicar todas las transformaciones y remocion de features a x_test tb?

In [ ]:
# Aplicar el filtro a X_train
X_train = X_train[seleccionadas].copy()
print(X_train.shape)

In [ ]:
X_train.head()

#### En el siguiente paso vamos a escalar  los valores

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)

In [ ]:
X_train_scaled.head()

##### La variable target `host_is_superhost` presenta una distribución de 57.7% (0) y 42.3% (1), lo cual representa un dataset prácticamente balanceado. Por lo tanto, 
##### no es necesario aplicar técnicas de balanceo (SMOTE, undersampling, etc.). Adicionalmente, se utilizó `stratify=y` en el train/test split para preservar 
##### esta proporción tanto en el conjunto de entrenamiento como en el de test.

Avanzamos con la reduccion de dimencionalidad PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Aplicar PCA sobre los datos escalados
pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)

# Varianza explicada por cada componente
varianza_explicada = pca.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza_explicada)

# Graficar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(varianza_explicada)+1), varianza_explicada, color='skyblue')
axes[0].set_title('Varianza explicada por componente')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Proporción de varianza')

axes[1].plot(range(1, len(varianza_acumulada)+1), varianza_acumulada, marker='o', color='teal')
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% varianza')
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% varianza')
axes[1].set_title('Varianza explicada acumulada')
axes[1].set_xlabel('Cantidad de componentes')
axes[1].set_ylabel('Varianza acumulada')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Componentes para 90% varianza: {np.argmax(varianza_acumulada >= 0.90) + 1}")
print(f"Componentes para 95% varianza: {np.argmax(varianza_acumulada >= 0.95) + 1}")

El análisis de PCA muestra que la varianza está distribuida de forma relativamente uniforme entre las componentes principales, requiriendo 
27-28 de las 30 componentes originales para retener el 95% de la varianza. Esto indica baja redundancia entre las variables tras la selección de features 
previamente realizada, por lo que PCA no resulta una técnica efectiva para reducir significativamente la dimensionalidad en este caso sin una pérdida 
considerable de información.

#### Ahora vamos a realizar la transfomaciones del X_test

Primero vamos a eliminar la variable 'has_availability'

In [ ]:
X_test = X_test.drop(columns=['has_availability']).copy()

La siguiente variable host_identity_verified que vamos a codificar con 0 y 1 es

In [ ]:
X_test['host_identity_verified'] = X_test['host_identity_verified'].map({'t': 1, 'f': 0})

Vamos a hacer el merge entre X_test y el csv de comunas

In [ ]:
X_test = X_test.merge(comunas, left_on='neighbourhood_cleansed', right_on='barrio', how='left')
X_test = X_test.drop(columns=['neighbourhood_cleansed', 'barrio']).copy()

Ahora vamos a realizar el OHE de las variables comuna y room_type

In [ ]:
ohe_test = ohe.transform(X_test[['comuna', 'room_type']])
ohe_cols = ohe.get_feature_names_out(['comuna', 'room_type'])
ohe_test_df = pd.DataFrame(ohe_test, columns=ohe_cols, index=X_test.index)

X_test = X_test.drop(columns=['comuna', 'room_type'])
X_test = pd.concat([X_test, ohe_test_df], axis=1)

Ahora vamos a imputar los nulos con el simple imputer ya fiteado

In [ ]:
columnas_imputar = ['review_scores_rating', 'review_scores_location', 
                     'minimum_nights', 'maximum_nights', 'bedrooms']
X_test[columnas_imputar] = imputer.transform(X_test[columnas_imputar])

Aplicamos el mismo filtro de P value que utilizamos en el X_train

In [ ]:
X_test = X_test[seleccionadas].copy()

Y por ultimo hacemos el escalamiento

In [ ]:
X_test_scaled = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

In [ ]:
X_test_scaled.shape

In [ ]:
X_test_scaled.head()

Dado que el análisis de PCA sobre `X_train` mostró que la varianza está distribuida de forma relativamente uniforme entre las componentes (requiriendo 
~27-28 de 30 componentes para retener el 95% de la varianza), se decidió **no aplicar PCA** como técnica de reducción de dimensionalidad. Esta decisión 
se mantiene de forma consistente para `X_test`, que conserva las mismas features seleccionadas que `X_train`.